# SARS-CoV-2: mutations against the reference and a tree of the genomes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fbcouto/multimerge-genomics-research/blob/main/notebooks/outbreaks/sars_cov_2_mutations_and_tree.ipynb)

**The problem.** You have consensus genomes of a virus (from an outbreak, a surveillance week or a
study). For each genome you need its mutations against the reference, named the way Nextclade,
GISAID and papers name them (`C241T`, `S:D614G`, `S:H69-`), and for the set, how the genomes relate:
a distance matrix and a tree to see clusters and lineages.

**What you need.** The genomes (FASTA, one record per genome); the reference genome (FASTA) and its
annotation (GFF3), for the amino-acid changes.

**What you get.** A table of mutations per genome (one TSV each, and a summary); the presence of the
mutations you watch across genomes; the pairwise distance matrix; a tree in Newick format and drawn.

In [ ]:
# Setup. Runs on Google Colab or on your own computer: on Colab, the first run installs MMGEN from PyPI
# (about a minute). Tables use pandas and plots matplotlib when installed (Colab has both); without
# them the notebook prints plain text and skips the plots.
import platform, subprocess, sys
from pathlib import Path

RELEASE = "https://github.com/fbcouto/multimerge-genomics-research/releases/download/v2.5.2"      # the demo packs
PACKAGE = "multimerge-genomics==2.5.2"   # from PyPI
try:
    import multimerge_genomics as mmg
except ImportError:
    if platform.system() != "Linux":
        print("MMGEN is not installed in this Python. Install it with:\n"
              "  python -m pip install multimerge-genomics\n"
              "and run this cell again.")
        raise
    print("Installing MMGEN (about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", PACKAGE], check=True)
    import multimerge_genomics as mmg

IN_NOTEBOOK = "ipykernel" in sys.modules
try:
    from IPython.display import display, Markdown
except ImportError:                        # plain Python: print instead
    display, Markdown = print, str
try:
    import pandas as pd
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)
except ImportError:
    pd = None
try:
    import matplotlib
    if not IN_NOTEBOOK:
        matplotlib.use("Agg")              # plain Python: save the figures, open no window
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

# Chart colors (a palette checked for color-blind readers): two series, a two-sided scale, neutral ink
BLUE, ORANGE, RED, GRAY, INK = "#2a78d6", "#eb6834", "#e34948", "#a3a29c", "#52514e"


def show(rows, n=25):
    """Shows a list of dictionaries as a table (pandas when available, plain text otherwise)."""
    rows = list(rows)
    if pd is not None:
        display(pd.DataFrame(rows[:n]))
    else:
        for row in rows[:n]:
            print("  ".join(f"{k}: {v}" for k, v in row.items()))
    if len(rows) > n:
        print(f"... and {len(rows) - n} more rows")


def tidy(ax):
    """Recessive axes: no top or right frame, a light grid behind the data."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)


def save_figure(fig, path):
    """Saves a figure; shows it in a notebook, closes it in plain Python."""
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Figure saved to", path)
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)


print(mmg.engine_info())

## Data

By default, from the research demo pack (downloaded once, about 45 MB): 20 complete SARS-CoV-2 genomes from NCBI
Virus, one per Pango lineage, from lineage A (January 2020) to KP.3.1.1 (2024), each record named
`<lineage>_<WHO label>_<accession>`; and the reference Wuhan-Hu-1 (NC_045512.2) with its annotation.
To use your own genomes, set `MY_GENOMES` (and the reference, for another virus).

In [ ]:
# Your genomes (None = the demo data)
MY_GENOMES = None            # a FASTA with one genome per record, or a list of FASTA files
MY_REFERENCE = None          # the reference genome (FASTA); default: Wuhan-Hu-1, NC_045512.2
MY_ANNOTATION = None         # its GFF3, for the amino-acid changes
OUTGROUP = "A_MN985325.1"    # the record to root the tree on (e.g. the oldest genome); None = midpoint
WATCH = ["ORF1b:P314L", "S:D614G", "S:N501Y", "S:E484K", "S:K417N", "S:L452R", "S:P681H", "S:P681R", "S:H69-"]

import urllib.request, zipfile

DATA = Path("data")                          # the demo files are extracted here
OUT = Path("results") / "sars_cov_2_mutations_and_tree"         # everything this notebook writes
OUT.mkdir(parents=True, exist_ok=True)


def demo_file(pack, member):
    """One file of a demo pack: downloads the pack once into the working folder and extracts only that file."""
    target = DATA / member
    if not target.exists():
        if not Path(pack).exists():
            print(f"Downloading {pack} from the MMGEN release...")
            urllib.request.urlretrieve(f"{RELEASE}/{pack}", pack + ".part")
            Path(pack + ".part").replace(pack)
        with zipfile.ZipFile(pack) as z:
            z.extract(member, DATA)
    return str(target)


USING_DEMO = MY_GENOMES is None
GENOMES = [MY_GENOMES] if isinstance(MY_GENOMES, str) else (MY_GENOMES or [demo_file("mmgen_demo_research_v4.zip", "research/outbreak/sars_cov_2_20_lineages.fasta")])
REFERENCE = MY_REFERENCE or demo_file("mmgen_demo_research_v4.zip", "research/outbreak/NC_045512.2.fna.gz")
ANNOTATION = MY_ANNOTATION or (demo_file("mmgen_demo_research_v4.zip", "research/outbreak/NC_045512.2.gff3.gz") if MY_REFERENCE is None else None)
print("Genomes:", GENOMES)
print("Reference:", REFERENCE, "| annotation:", ANNOTATION)

## Step 1. Mutations of each genome

Each genome is aligned to the reference and its differences are written in reference coordinates:
substitutions, deletions, insertions, and stretches of `N` (not sequenced: not a mutation, but a mutation
there cannot be seen). With the annotation, each change is translated into its amino-acid change in the
gene it falls in (ORF1ab reported as ORF1a and ORF1b, as Nextclade does). Low coverage of the reference or
long `N` stretches mean a poor consensus: interpret those genomes with care.

In [ ]:
vm = mmg.viral_mutations(REFERENCE, GENOMES, annotation=ANNOTATION, output_dir=str(OUT / "per_genome"),
                         summary_output=str(OUT / "mutations_summary.tsv"))
for w in vm["warnings"]:
    print("Note:", w)
show(({"genome": g["name"], "reference covered %": round(100 * g["coverage"], 2),
       "substitutions": len(g["substitutions"]), "deletions": len(g["deletions"]), "insertions": len(g["insertions"]),
       "N stretches": len(g["missing"]),
       "amino-acid changes": len(g["aa_substitutions"]) + len(g["aa_deletions"]) + len(g["aa_insertions"]),
       "problem": g["problem"] or ""} for g in vm["genomes"]), n=50)
print("Summary table:", OUT / "mutations_summary.tsv", "| one table per genome in", OUT / "per_genome")

## Step 2. The mutations you watch

Which genomes carry each mutation of `WATCH`: here mutations that marked the variants of concern
(D614G spread in early 2020; N501Y in Alpha, Beta, Gamma and Omicron; L452R and P681R in Delta).
A dot means not found, which includes "in an `N` stretch".

Look at the demo's Alpha genome: it has Alpha's spike changes A570D, P681H, T716I and S982A, but not N501Y nor
the 69-70 and 144 deletions. A consensus can lack mutations its lineage should have (for example where an
amplicon dropped out and the gap was filled with the reference instead of `N`): check surprising absences
against the reads and their coverage before drawing conclusions.

In [ ]:
def aa_changes(g):
    return {m["label"] for key in ("aa_substitutions", "aa_deletions", "aa_insertions") for m in g[key]}


changes = {g["name"]: aa_changes(g) for g in vm["genomes"]}
show(({"genome": name, **{m: "+" if m in found else "." for m in WATCH}} for name, found in changes.items()), n=50)
print("Carriers:", {m: sum(m in found for found in changes.values()) for m in WATCH})

## Step 3. Distances and a tree

Each genome is reduced to its 31-base k-mers (a FracMinHash sketch, the same as sourmash; for genomes this
small every k-mer is kept). Two genomes' shared k-mers give the Mash distance and an average nucleotide
identity (ANI). For an outbreak, the count of nucleotide differences between two genomes (from step 1) is
the usual measure: a few differences fit direct transmission, many do not.

Neighbor joining on the distance matrix, rooted on `OUTGROUP` (an early genome, so the tree reads from
past to present) or, without one, at the midpoint of the longest path. Saved in Newick format, which
iTOL, FigTree, MEGA or ete3 open. A k-mer distance tree is a quick look; for a publication, build a
maximum-likelihood tree from an alignment (IQ-TREE, UShER).

In [ ]:
names = [g["name"] for g in vm["genomes"]]
outgroup = OUTGROUP if OUTGROUP in names else None
dist = mmg.genome_distances(GENOMES, records_as_genomes=True, outgroup=outgroup,
                            matrix_output=str(OUT / "distances.tsv"), tree_output=str(OUT / "tree.nwk"))
subs = {g["name"]: {m["label"] for m in g["substitutions"]} for g in vm["genomes"]}
rows = []
for i, a in enumerate(dist["names"]):
    j = max((j for j in range(len(dist["names"])) if j != i), key=lambda j: dist["ani"][i][j])
    b = dist["names"][j]
    rows.append({"genome": a, "closest": b, "ANI %": round(100 * dist["ani"][i][j], 3),
                 "Mash distance": round(dist["distance"][i][j], 5),
                 "substitutions that differ": len(subs[a] ^ subs[b]) if a in subs and b in subs else None})
show(rows, n=50)
print("Distance matrix:", OUT / "distances.tsv")

tree = dist["tree"]
nodes = {n["id"]: n for n in tree["nodes"]}
print(tree["newick"][:300] + ("..." if len(tree["newick"]) > 300 else ""))
print(f"Rooted on {outgroup or 'the midpoint'}; Newick written to", OUT / "tree.nwk")

if plt is not None:
    leaves = [n for n in nodes.values() if not n["children"]]
    fig, ax = plt.subplots(figsize=(9, 0.32 * len(leaves) + 1))
    for n in nodes.values():
        if n["parent"] is not None:
            ax.plot([nodes[n["parent"]]["x"], n["x"]], [n["y"], n["y"]], color=INK, linewidth=1.2)
        if n["children"]:
            ys = [nodes[c]["y"] for c in n["children"]]
            ax.plot([n["x"], n["x"]], [min(ys), max(ys)], color=INK, linewidth=1.2)
        else:
            ax.text(n["x"], n["y"], "  " + n["name"], va="center", fontsize=8)
    ax.invert_yaxis()
    ax.set_yticks([])
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.set_xlabel("Mash distance from the root")
    ax.set_xlim(0, tree["depth"] * 1.45)
    ax.set_title(f"Neighbor-joining tree of {len(leaves)} genomes")
    save_figure(fig, OUT / "tree.png")

## Check the result

In [ ]:
def check(condition, text):
    print(("OK     " if condition else "CHECK  ") + text)


def ancestors(node_id):
    chain = [node_id]
    while nodes[chain[-1]]["parent"] is not None:
        chain.append(nodes[chain[-1]]["parent"])
    return chain


def is_clade(group):
    """True when the genomes of `group` (and only them) descend from their most recent common ancestor."""
    leaf = {n["name"]: n["id"] for n in nodes.values() if not n["children"]}
    chains = [ancestors(leaf[name]) for name in group]
    mrca = next(a for a in chains[0] if all(a in c for c in chains[1:]))
    below = {n["name"] for n in nodes.values() if not n["children"] and mrca in ancestors(n["id"])}
    return below == set(group)


check(all(g["aligned"] for g in vm["genomes"]), "every genome aligned to the reference")
check(all(g["coverage"] > 0.95 for g in vm["genomes"]), "every genome covers more than 95% of the reference")
check(len(dist["names"]) == len(vm["genomes"]), "the tree has every genome")
if USING_DEMO:
    after_b1 = [n for n in changes if not n.startswith("A_")]
    assert all("S:D614G" in changes[n] for n in after_b1) and "S:D614G" not in changes["A_MN985325.1"]
    assert all("S:N501Y" in changes[n] for n in changes if any(w in n for w in ("Beta", "Gamma", "Mu", "Omicron")))
    assert is_clade([n for n in names if "Delta" in n]), "the two Delta genomes should be neighbors"
    assert is_clade([n for n in names if "Omicron" in n]), "the Omicron genomes should form one clade"
    print("Demo expectations met: D614G after lineage A, N501Y in Beta/Gamma/Mu/Omicron, Delta and Omicron clades.")

## Methods paragraph

A starting text for the methods section of a paper or report, filled in with the version and the parameters
this run used. Edit it to your study, and cite the methods MMGEN reproduces (see References).

In [ ]:
import textwrap
VERSION = mmg.engine_info()["version"]
methods = f"""{len(vm['genomes'])} genomes were compared with the reference {Path(REFERENCE).name} using viral_mutations,
which aligns each genome to the reference and names nucleotide and amino-acid changes as Nextclade does (ORF1ab
reported as ORF1a and ORF1b). Pairwise distances were estimated from FracMinHash sketches (k = {dist['k']},
scaled = {dist['scaled']}; Mash distance), and a neighbor-joining tree was built and rooted on
{outgroup or 'the midpoint'} with genome_distances. All steps used MMGEN version {VERSION} (Couto F. MMGEN: MultiMerge Genomics (version {VERSION}). Zenodo. https://doi.org/10.5281/zenodo.20417492)."""
print(textwrap.fill(" ".join(methods.split()), 110))

## Adapt it to your data

* Another virus: give its RefSeq genome and GFF3 (`MY_REFERENCE`, `MY_ANNOTATION`) and your own `WATCH` list.
* Genomes in separate files: `MY_GENOMES = ["s1.fasta", "s2.fasta", ...]` (one record per genome).
* Bacterial isolates: `genome_distances` takes one FASTA (or a list of files) per genome without
  `records_as_genomes`; `scaled=1000` keeps it fast.
* Assign Pango lineages with Pangolin or Nextclade; MMGEN names mutations but does not assign lineages.
* Look at the per-genome TSVs in `results/.../per_genome` for every change with its position and gene.

## References

* Aksamentov I. et al. (2021) Nextclade: clade assignment, mutation calling and quality control for viral
  genomes. *Journal of Open Source Software* 6:3773.
* Rambaut A. et al. (2020) A dynamic nomenclature proposal for SARS-CoV-2 lineages. *Nature Microbiology* 5:1403.
* Wu F. et al. (2020) A new coronavirus associated with human respiratory disease in China. *Nature* 579:265
  (the reference genome).
* Ondov B.D. et al. (2016) Mash: fast genome and metagenome distance estimation using MinHash. *Genome Biology* 17:132.
* Irber L. et al. (2022) Lightweight compositional analysis of metagenomes with FracMinHash and minimum metagenome
  covers. *bioRxiv* 2022.01.11.475838 (sourmash).
* Saitou N., Nei M. (1987) The neighbor-joining method. *Molecular Biology and Evolution* 4:406.